selection + pca(100) + lag 

In [5]:
import sys

!{sys.executable} -m pip uninstall -y pandas pyarrow
!{sys.executable} -m pip install --no-cache-dir pandas pyarrow

Found existing installation: pandas 3.0.1
Uninstalling pandas-3.0.1:
  Successfully uninstalled pandas-3.0.1
Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 2.3 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 2.0 MB/s  0:00:17m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd

train_path = "/Users/seomichelle/26-1 ESAA:Python/datasets/프로젝트/train.parquet"

train = pd.read_parquet(train_path)

train.head()

,era,data_type,feature_shaded_hallucinatory_dactylology,feature_itinerant_hexahedral_photoengraver,feature_prudent_pileate_oven,feature_subalpine_apothegmatical_ajax,feature_pistachio_atypical_malison,feature_symmetrical_spongy_tricentenary,feature_ungrounded_transpontine_winder,feature_aseptic_eely_hemiplegia,...,target_teager2b_20,target_teager2b_60,target_tyler_20,target_tyler_60,target_victor_20,target_victor_60,target_waldo_20,target_waldo_60,target_xerxes_20,target_xerxes_60
id,,,,,,,,,,,,,,,,,,,,,
n0007b5abb0c3a25,0001,train,3,4,0,3,3,1,1,0,...,0.50,0.50,0.25,0.25,0.25,0.25,0.25,0.00,0.25,0.00
n003bba8a98662e4,0001,train,4,2,4,4,0,0,4,4,...,0.50,0.50,0.25,0.25,0.25,0.00,0.25,0.25,0.25,0.25
n003bee128c2fcfc,0001,train,2,4,0,3,0,3,2,4,...,1.00,1.00,1.00,0.75,0.75,0.75,0.75,1.00,0.75,0.75
n0048ac83aff7194,0001,train,2,1,3,0,3,0,3,3,...,0.25,0.25,0.25,0.25,0.50,0.25,0.25,0.25,0.25,0.25
n0055a2401ba6480,0001,train,4,1,4,1,0,4,0,4,...,0.50,0.50,0.25,0.50,0.25,0.50,0.25,0.50,0.25,0.50


In [3]:
import numpy as np
import pandas as pd

feature_cols = [col for col in train.columns if col.startswith("feature_")]
target_cols = [col for col in train.columns if col.startswith("target_")]

print("feature 개수:", len(feature_cols))
print("target 개수:", len(target_cols))

feature 개수: 2748
target 개수: 40


In [ ]:
# =========================
# 0. Library
# =========================

import sys
import subprocess
import numpy as np
import pandas as pd
import lightgbm as lgb
import lightgbm as lgb

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA
from sklearn.metrics import mean_squared_error, mean_absolute_error


# =========================
# 1. 기본 설정
# =========================

TARGET_COL = "target_tyler_20"   # 원하는 target으로 변경 가능
SELECTION_K = 500                # feature selection 후 남길 feature 수
PCA_N = 100                      # PCA component 개수
LAG_FEATURE_N = 20               # lag로 만들 feature 개수
LAGS = [1]                       # 이전 era 1개 lag 사용
CHUNK_SIZE = 50000               # 메모리 절약용, 샘플링 아님

RANDOM_STATE = 42


# =========================
# 2. feature / target 컬럼 분리
# =========================

feature_cols = [col for col in train.columns if col.startswith("feature_")]
target_cols = [col for col in train.columns if col.startswith("target_")]

if TARGET_COL not in target_cols:
    TARGET_COL = target_cols[0]

print("전체 데이터 크기:", train.shape)
print("feature 개수:", len(feature_cols))
print("target 개수:", len(target_cols))
print("사용 target:", TARGET_COL)


# =========================
# 3. 학습용 데이터 정리
# =========================

df = train.copy()

df = df[df[TARGET_COL].notna()].copy()
df["era_int"] = df["era"].astype(int)
df = df.sort_values("era_int").copy()

print("모델링 데이터 크기:", df.shape)


# =========================
# 4. era 기준 train / validation split
# =========================

unique_eras = sorted(df["era"].unique(), key=lambda x: int(x))
split_point = int(len(unique_eras) * 0.8)

train_eras = unique_eras[:split_point]
valid_eras = unique_eras[split_point:]

train_mask = df["era"].isin(train_eras).values
valid_mask = df["era"].isin(valid_eras).values

train_positions = np.where(train_mask)[0]
valid_positions = np.where(valid_mask)[0]

print("train era 개수:", len(train_eras))
print("valid era 개수:", len(valid_eras))
print("train rows:", len(train_positions))
print("valid rows:", len(valid_positions))




전체 데이터 크기: (2746268, 2791)
feature 개수: 2748
target 개수: 40
사용 target: target_tyler_20
모델링 데이터 크기: (2746268, 2792)
train era 개수: 459
valid era 개수: 115
train rows: 2158685
valid rows: 587583


In [ ]:
# =========================
# 5. Feature Selection
# 각 feature와 target의 상관관계를 계산한 뒤 상위 K개 선택

def select_features_all_rows(df, feature_cols, target_col, positions, k=500, chunk_size=50000):
    n_total = 0

    p = len(feature_cols)

    sum_x = np.zeros(p, dtype=np.float64)
    sum_x2 = np.zeros(p, dtype=np.float64)
    sum_xy = np.zeros(p, dtype=np.float64)

    sum_y = 0.0
    sum_y2 = 0.0

    for start in range(0, len(positions), chunk_size):
        end = min(start + chunk_size, len(positions))
        pos_chunk = positions[start:end]

        X_chunk = df.iloc[pos_chunk][feature_cols].to_numpy(dtype=np.float32)
        y_chunk = df.iloc[pos_chunk][target_col].to_numpy(dtype=np.float32)

        X_chunk = np.nan_to_num(X_chunk, nan=0.0, posinf=0.0, neginf=0.0)
        y_chunk = np.nan_to_num(y_chunk, nan=0.0, posinf=0.0, neginf=0.0)

        n_chunk = len(y_chunk)
        n_total += n_chunk

        sum_x += X_chunk.sum(axis=0, dtype=np.float64)
        sum_x2 += (X_chunk * X_chunk).sum(axis=0, dtype=np.float64)
        sum_xy += X_chunk.T @ y_chunk.astype(np.float64)

        sum_y += y_chunk.sum(dtype=np.float64)
        sum_y2 += (y_chunk * y_chunk).sum(dtype=np.float64)

        print(f"Feature selection 진행: {end} / {len(positions)}")

    cov_xy = sum_xy - (sum_x * sum_y / n_total)
    var_x = sum_x2 - (sum_x ** 2 / n_total)
    var_y = sum_y2 - (sum_y ** 2 / n_total)

    corr = cov_xy / (np.sqrt(var_x * var_y) + 1e-12)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)

    scores = np.abs(corr)

    k = min(k, len(feature_cols))
    selected_idx = np.argsort(scores)[-k:][::-1]

    selected_features = [feature_cols[i] for i in selected_idx]

    score_df = pd.DataFrame({
        "feature": feature_cols,
        "abs_corr_score": scores
    }).sort_values("abs_corr_score", ascending=False)

    return selected_features, score_df


selected_features, feature_score_df = select_features_all_rows(
    df=df,
    feature_cols=feature_cols,
    target_col=TARGET_COL,
    positions=train_positions,
    k=SELECTION_K,
    chunk_size=CHUNK_SIZE
)

print("선택된 feature 개수:", len(selected_features))
print("선택된 feature 상위 10개:")
display(feature_score_df.head(10))


# =========================
# 6. Scaling fit
# 전체 train 데이터를 모두 사용해서 StandardScaler fit


scaler = StandardScaler()

for start in range(0, len(train_positions), CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, len(train_positions))
    pos_chunk = train_positions[start:end]

    X_chunk = df.iloc[pos_chunk][selected_features].to_numpy(dtype=np.float32)
    X_chunk = np.nan_to_num(X_chunk, nan=0.0, posinf=0.0, neginf=0.0)

    scaler.partial_fit(X_chunk)

    print(f"Scaler fit 진행: {end} / {len(train_positions)}")


# =========================
# 7. PCA 100 fit
# 전체 train 데이터를 모두 사용해서 Incremental PCA fit


PCA_N = min(PCA_N, len(selected_features))

pca = IncrementalPCA(
    n_components=PCA_N,
    batch_size=CHUNK_SIZE
)

for start in range(0, len(train_positions), CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, len(train_positions))
    pos_chunk = train_positions[start:end]

    X_chunk = df.iloc[pos_chunk][selected_features].to_numpy(dtype=np.float32)
    X_chunk = np.nan_to_num(X_chunk, nan=0.0, posinf=0.0, neginf=0.0)

    X_chunk_scaled = scaler.transform(X_chunk)

    if X_chunk_scaled.shape[0] >= PCA_N:
        pca.partial_fit(X_chunk_scaled)

    print(f"PCA fit 진행: {end} / {len(train_positions)}")

print("PCA component 개수:", PCA_N)
print("PCA 설명 분산 비율 합:", pca.explained_variance_ratio_.sum())


Feature selection 진행: 50000 / 2158685
Feature selection 진행: 100000 / 2158685
Feature selection 진행: 150000 / 2158685
Feature selection 진행: 200000 / 2158685
Feature selection 진행: 250000 / 2158685
Feature selection 진행: 300000 / 2158685
Feature selection 진행: 350000 / 2158685
Feature selection 진행: 400000 / 2158685
Feature selection 진행: 450000 / 2158685
Feature selection 진행: 500000 / 2158685
Feature selection 진행: 550000 / 2158685
Feature selection 진행: 600000 / 2158685
Feature selection 진행: 650000 / 2158685
Feature selection 진행: 700000 / 2158685
Feature selection 진행: 750000 / 2158685
Feature selection 진행: 800000 / 2158685
Feature selection 진행: 850000 / 2158685
Feature selection 진행: 900000 / 2158685
Feature selection 진행: 950000 / 2158685
Feature selection 진행: 1000000 / 2158685
Feature selection 진행: 1050000 / 2158685
Feature selection 진행: 1100000 / 2158685
Feature selection 진행: 1150000 / 2158685
Feature selection 진행: 1200000 / 2158685
Feature selection 진행: 1250000 / 2158685
Feature selection 진행

,feature,abs_corr_score
1118,feature_jewish_stained_disembowelment,0.017801
1103,feature_deprivable_prothalloid_rodrigo,0.016986
1117,feature_meridional_synonymic_wyatt,0.016668
1102,feature_waterlog_gradual_smallage,0.015658
1120,feature_exclusionist_enwrapped_sullage,0.015433
1116,feature_peristomatic_unneighbourly_dogeship,0.014740
1105,feature_weest_romanic_bornite,0.014523
1119,feature_conglobate_scolopendrine_wordsworth,0.014184
1098,feature_anamorphic_hidden_termagant,0.014058
1101,feature_interoceptive_stodgy_monostich,0.013997


Scaler fit 진행: 50000 / 2158685
Scaler fit 진행: 100000 / 2158685
Scaler fit 진행: 150000 / 2158685
Scaler fit 진행: 200000 / 2158685
Scaler fit 진행: 250000 / 2158685
Scaler fit 진행: 300000 / 2158685
Scaler fit 진행: 350000 / 2158685
Scaler fit 진행: 400000 / 2158685
Scaler fit 진행: 450000 / 2158685
Scaler fit 진행: 500000 / 2158685
Scaler fit 진행: 550000 / 2158685
Scaler fit 진행: 600000 / 2158685
Scaler fit 진행: 650000 / 2158685
Scaler fit 진행: 700000 / 2158685
Scaler fit 진행: 750000 / 2158685
Scaler fit 진행: 800000 / 2158685
Scaler fit 진행: 850000 / 2158685
Scaler fit 진행: 900000 / 2158685
Scaler fit 진행: 950000 / 2158685
Scaler fit 진행: 1000000 / 2158685
Scaler fit 진행: 1050000 / 2158685
Scaler fit 진행: 1100000 / 2158685
Scaler fit 진행: 1150000 / 2158685
Scaler fit 진행: 1200000 / 2158685
Scaler fit 진행: 1250000 / 2158685
Scaler fit 진행: 1300000 / 2158685
Scaler fit 진행: 1350000 / 2158685
Scaler fit 진행: 1400000 / 2158685
Scaler fit 진행: 1450000 / 2158685
Scaler fit 진행: 1500000 / 2158685
Scaler fit 진행: 1550000 / 21586

In [ ]:
# =========================
# 8. 전체 데이터에 PCA 변환 적용
# =========================

def transform_pca_all_rows(df, selected_features, scaler, pca, chunk_size=50000):
    pca_cols = [f"pca_{i+1}" for i in range(pca.n_components_)]
    pca_array = np.zeros((len(df), pca.n_components_), dtype=np.float32)

    all_positions = np.arange(len(df))

    for start in range(0, len(all_positions), chunk_size):
        end = min(start + chunk_size, len(all_positions))
        pos_chunk = all_positions[start:end]

        X_chunk = df.iloc[pos_chunk][selected_features].to_numpy(dtype=np.float32)
        X_chunk = np.nan_to_num(X_chunk, nan=0.0, posinf=0.0, neginf=0.0)

        X_chunk_scaled = scaler.transform(X_chunk)
        X_chunk_pca = pca.transform(X_chunk_scaled).astype(np.float32)

        pca_array[start:end, :] = X_chunk_pca

        print(f"PCA transform 진행: {end} / {len(all_positions)}")

    return pd.DataFrame(
        pca_array,
        index=df.index,
        columns=pca_cols
    )


pca_df = transform_pca_all_rows(
    df=df,
    selected_features=selected_features,
    scaler=scaler,
    pca=pca,
    chunk_size=CHUNK_SIZE
)

print("PCA 데이터 크기:", pca_df.shape)


# =========================
# 9. Lag Feature 생성
# era별 selected feature 평균의 lag 사용

lag_base_features = selected_features[:LAG_FEATURE_N]

era_means = (
    df.groupby("era")[lag_base_features]
    .mean()
    .reindex(unique_eras)
)

lag_dfs = []

for lag in LAGS:
    lagged = era_means.shift(lag)
    lagged.columns = [f"lag{lag}_era_mean_{col}" for col in lagged.columns]
    lag_dfs.append(lagged)

lag_by_era = pd.concat(lag_dfs, axis=1).fillna(0)

lag_df = df[["era"]].merge(
    lag_by_era,
    left_on="era",
    right_index=True,
    how="left"
)

lag_df.index = df.index
lag_df = lag_df.drop(columns=["era"]).astype(np.float32)

print("lag feature 크기:", lag_df.shape)


# =========================
# 10. 최종 학습 데이터 생성
# =========================
# 최종 feature = PCA 100개 + lag feature
# =========================

X_all = pd.concat([pca_df, lag_df], axis=1).astype(np.float32)
y_all = df[TARGET_COL].astype(np.float32)

X_train = X_all.iloc[train_positions]
y_train = y_all.iloc[train_positions]

X_valid = X_all.iloc[valid_positions]
y_valid = y_all.iloc[valid_positions]

print("최종 X_train 크기:", X_train.shape)
print("최종 X_valid 크기:", X_valid.shape)



PCA transform 진행: 50000 / 2746268
PCA transform 진행: 100000 / 2746268
PCA transform 진행: 150000 / 2746268
PCA transform 진행: 200000 / 2746268
PCA transform 진행: 250000 / 2746268
PCA transform 진행: 300000 / 2746268
PCA transform 진행: 350000 / 2746268
PCA transform 진행: 400000 / 2746268
PCA transform 진행: 450000 / 2746268
PCA transform 진행: 500000 / 2746268
PCA transform 진행: 550000 / 2746268
PCA transform 진행: 600000 / 2746268
PCA transform 진행: 650000 / 2746268
PCA transform 진행: 700000 / 2746268
PCA transform 진행: 750000 / 2746268
PCA transform 진행: 800000 / 2746268
PCA transform 진행: 850000 / 2746268
PCA transform 진행: 900000 / 2746268
PCA transform 진행: 950000 / 2746268
PCA transform 진행: 1000000 / 2746268
PCA transform 진행: 1050000 / 2746268
PCA transform 진행: 1100000 / 2746268
PCA transform 진행: 1150000 / 2746268
PCA transform 진행: 1200000 / 2746268
PCA transform 진행: 1250000 / 2746268
PCA transform 진행: 1300000 / 2746268
PCA transform 진행: 1350000 / 2746268
PCA transform 진행: 1400000 / 2746268
PCA transfor

In [9]:
# =========================
# 11. LightGBM 모델 학습
# =========================

model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="l2",
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=100)
    ]
)


# =========================
# 12. Validation 예측 및 성능 확인
# =========================

valid_pred = model.predict(X_valid)

rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
mae = mean_absolute_error(y_valid, valid_pred)

print("\nValidation RMSE:", rmse)
print("Validation MAE:", mae)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.311279 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 27183
[LightGBM] [Info] Number of data points in the train set: 2158685, number of used features: 120
[LightGBM] [Info] Start training from score 0.499947
Training until validation scores don't improve for 100 rounds
[100]	valid_0's l2: 0.0499544
[200]	valid_0's l2: 0.0499391
[300]	valid_0's l2: 0.0499379
[400]	valid_0's l2: 0.0499413
Early stopping, best iteration is:
[312]	valid_0's l2: 0.0499375

Validation RMSE: 0.2234669407067369
Validation MAE: 0.15332256493891422
